In [1]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv

In [2]:
def extract_valid_datasets(block):
    datasets = []
    if isinstance(block, pv.MultiBlock):
        for i in range(block.n_blocks):
            datasets.extend(extract_valid_datasets(block[i]))
    else:
        if block is None:
            return []
        if block.n_points == 0 or block.n_cells == 0:
            return []
        
        datasets.append(block)
    return datasets

def debug_blocks(block, level=0):
    indent = "  " * level
    if isinstance(block, pv.MultiBlock):
        print(f"{indent}MultiBlock with {block.n_blocks} blocks")
        for i in range(block.n_blocks):
            debug_blocks(block[i], level + 1)
    else:
        print(f"{indent}Dataset: {block}, Points: {block.n_points}, Cells: {block.n_cells}")

def load_mesh(filename):
    reader = pv.get_reader(filename)
    reader.set_active_time_value(reader.time_values[-1])
    mb = reader.read()

    valid_blocks = extract_valid_datasets(mb)
    mesh = pv.MultiBlock(valid_blocks).combine()

    # Convert cell data to point data
    mesh = mesh.cell_data_to_point_data()

    # Print the available array names and point data keys
    print(f"Analyzing mesh from file: {filename.split('/')[-1]}")
    print(f"{mesh.n_points} points")
    print(f"Array Names: {mesh.array_names}")
    print(f"Point Data Keys: {list(mesh.point_data.keys())}")    
    return mesh

In [3]:
# Load the file
filename_rd1 = "../results/rd0001.e"
mesh_rd1 = load_mesh(filename_rd1)

Analyzing mesh from file: rd0001.e
19402 points
Array Names: ['ElementBlockIds', 'Title', 'Info_Records', 'Pe', 'T', 'p', 'vel_', 'ObjectId']
Point Data Keys: ['Pe', 'T', 'p', 'vel_', 'ObjectId']


In [4]:
coords = mesh_rd1.points

T = mesh_rd1["T"]
p = mesh_rd1["p"]
vel = mesh_rd1["vel_"]
u = vel[:, 0]
v = vel[:, 1]

print("Coords shape:", coords.shape)
print("T shape:", T.shape)
print("p shape:", p.shape)
print("vel shape:", vel.shape)

Coords shape: (19402, 3)
T shape: (19402,)
p shape: (19402,)
vel shape: (19402, 3)


In [5]:
print("Any NaNs in T?", np.isnan(T).any())
print("Any NaNs in p?", np.isnan(p).any())
print("Any NaNs in vel?", np.isnan(vel).any())

print("T range:", T.min(), T.max())
print("p range:", p.min(), p.max())

Any NaNs in T? False
Any NaNs in p? False
Any NaNs in vel? False
T range: 0.0 1.0000000000000004
p range: -289.37449557969757 288778.4560511


In [6]:
# Code to make sure the meshes are the same for multiple files
filename_rd2 = "../results/hx.e"
mesh_rd2 = load_mesh(filename_rd2)
same_grid = np.allclose(mesh_rd1.points, mesh_rd2.points)
print("Same grid?", same_grid)


Analyzing mesh from file: hx.e
31066 points
Array Names: ['ElementBlockIds', 'Title', 'Info_Records', 'Pe', 'T', 'p', 'vel_', 'ObjectId']
Point Data Keys: ['Pe', 'T', 'p', 'vel_', 'ObjectId']


ValueError: operands could not be broadcast together with shapes (19402,3) (31066,3) 

In [7]:
# Build a reference grid for interpolation
xmin, xmax, ymin, ymax, zmin, zmax = mesh_rd1.bounds
nx, ny = 100, 50

x = np.linspace(xmin, xmax, nx)
y = np.linspace(ymin, ymax, ny)
z = np.array([0])  # Assuming 2D data in the xy-plane
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

grid = pv.StructuredGrid()
grid.points = np.c_[X.ravel(), Y.ravel(), Z.ravel()]
grid.dimensions = [nx, ny, 1]

In [8]:
# Sample the CFD data onto the reference grid
sampled = grid.sample(mesh_rd1)
T_grid = sampled["T"]
p_grid = sampled["p"]
vel_grid = sampled["vel_"]

In [9]:
print(T_grid)
for t in T_grid:
    print(t)

[0. 0. 0. ... 1. 1. 1.]
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
5.4820852339070505e-05
4.931168882103738e-05
4.063448762566293e-05
3.1636994437267267e-05
2.3796485348149936e-05
1.7520784553692396e-05
1.2869555416171727e-05
9.43886456241892e-06
6.848698437842784e-06
4.9275870714261925e-06
3.5281207415075704e-06
2.5356866561918547e-06
1.8827983585503262e-06
1.4147663124634572e-06
1.0875105548741138e-06
8.652731754891985e-07
7.198005388831292e-07
6.354890987407216e-07
5.864087643361203e-07
5.590183643611038e-07
5.463069848644573e-07
5.427363978274815e-07
5.441048078259208e-07
5.466273321966661e-07
5.481679823901968e-07
5.481682783350349e-07
5.466282337342953e-07
5.441063566100255e-07
5.427386988753119e-07
5.463101987058718e-07
5.590227156500717e-07
5.864145680043559e-07
6.354967839636681e-07
7.198107633832236e-07
8.652870280053845e

In [10]:
# Identify Solid vs Fluid and fix NaN values
mask = ~np.isnan(T_grid)
mask = mask.astype(float)
print("Fluid points:", np.sum(mask))
print("Solid points:", np.sum(mask == 0))

# Clean the data
T_grid = np.nan_to_num(T_grid, nan=0.0)
p_grid = np.nan_to_num(p_grid, nan=0.0)
vel_grid = np.nan_to_num(vel_grid, nan=0.0)

Fluid points: 5000.0
Solid points: 0


In [11]:
# Build the machine learning dataset
coords = sampled.points
u = vel_grid[:, 0]
v = vel_grid[:, 1]

X_data = np.column_stack([
    coords[:, 0],
    coords[:, 1],
    mask
])

Y_data = np.column_stack([
    u,
    v,
    p_grid,
    T_grid
])

In [12]:
print(np.isnan(T_grid).sum())

0


In [13]:
import torch
density_np = np.linspace(0, 1, 10)
density_tensor = torch.from_numpy(density_np)
too_low = torch.relu(0.25 - density_tensor)
too_high = torch.relu(density_tensor - 0.70)
print("Density:", density_tensor)
print("Too low:", too_low)
print("Too high:", too_high)
print((too_low + too_high).mean())

Density: tensor([0.0000, 0.1111, 0.2222, 0.3333, 0.4444, 0.5556, 0.6667, 0.7778, 0.8889,
        1.0000], dtype=torch.float64)
Too low: tensor([0.2500, 0.1389, 0.0278, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000], dtype=torch.float64)
Too high: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0778, 0.1889,
        0.3000], dtype=torch.float64)
tensor(0.0983, dtype=torch.float64)
